### Testing

In [7]:
import pandas as pd
import json
from pathlib import Path
import os
import pytz
from datetime import datetime

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
# In a notebook, use cwd() (Current Working Directory) instead of __file__
BASE_DIR = Path.cwd().parent
RAW_DATA_PATH = os.path.join(BASE_DIR, 'data', 'raw')

# Verify the path
print(f"Reading from: {RAW_DATA_PATH}")

for file in os.listdir(RAW_DATA_PATH):
    with open(os.path.join(RAW_DATA_PATH, file), 'r') as f:
        data = json.load(f)
        df = pd.json_normalize(data)
        print(f'file: {file}')

Reading from: c:\Users\annab\beatrizbeserra\PROJETOS\GITHUB\Portifolio\projetos\data_engineering\credit_guard_ml_pipeline\data\raw
file: 00000000000191_2026-02-27.json
file: 00001180000126_2026-02-27.json
file: 00360305000104_2026-02-27.json
file: 00776574000156_2026-02-27.json
file: 00864214000106_2026-02-27.json
file: 01027058000191_2026-02-27.json
file: 01637895000132_2026-02-27.json
file: 02012862000160_2026-02-27.json
file: 02332886000104_2026-02-27.json
file: 02429144000193_2026-02-27.json
file: 02916265000160_2026-02-27.json
file: 03220438000173_2026-02-27.json
file: 03361252000134_2026-02-27.json
file: 03853896000140_2026-02-27.json
file: 05197443000138_2026-02-27.json
file: 05878397000132_2026-02-27.json
file: 07526557000100_2026-02-27.json
file: 07575651000159_2026-02-27.json
file: 07707650000110_2026-02-27.json
file: 08561701000101_2026-02-27.json
file: 09296295000160_2026-02-27.json
file: 09346601000125_2026-02-27.json
file: 14380200000121_2026-02-27.json
file: 156646490001

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 48 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   uf                                     1 non-null      str   
 1   cep                                    1 non-null      str   
 2   qsa                                    1 non-null      object
 3   cnpj                                   1 non-null      str   
 4   pais                                   0 non-null      object
 5   email                                  0 non-null      object
 6   porte                                  1 non-null      str   
 7   bairro                                 1 non-null      str   
 8   numero                                 1 non-null      str   
 9   ddd_fax                                1 non-null      str   
 10  municipio                              1 non-null      str   
 11  logradouro                        

In [11]:
df.head()

,uf,cep,qsa,cnpj,pais,email,porte,bairro,numero,ddd_fax,municipio,logradouro,cnae_fiscal,codigo_pais,complemento,codigo_porte,razao_social,nome_fantasia,capital_social,ddd_telefone_1,ddd_telefone_2,opcao_pelo_mei,codigo_municipio,cnaes_secundarios,natureza_juridica,regime_tributario,situacao_especial,opcao_pelo_simples,situacao_cadastral,data_opcao_pelo_mei,data_exclusao_do_mei,cnae_fiscal_descricao,codigo_municipio_ibge,data_inicio_atividade,data_situacao_especial,data_opcao_pelo_simples,data_situacao_cadastral,nome_cidade_no_exterior,codigo_natureza_juridica,data_exclusao_do_simples,motivo_situacao_cadastral,ente_federativo_responsavel,identificador_matriz_filial,qualificacao_do_responsavel,descricao_situacao_cadastral,descricao_tipo_de_logradouro,descricao_motivo_situacao_cadastral,descricao_identificador_matriz_filial
0,SP,05426100,"[{'pais': None, 'nome_socio': 'ALBERTO KLABIN'...",89637490000145,None,None,DEMAIS,PINHEIROS,949,1130465846,SAO PAULO,BRIG FARIA LIMA,1710900,None,ANDAR 12-14-15-16 CONJ 1602,5,KLABIN S.A.,,6075625000,1130469918,,None,7107,"[{'codigo': 8211300, 'descricao': 'ServiÃ§os c...",Sociedade AnÃ´nima Aberta,"[{'ano': 2014, 'cnpj_da_scp': None, 'forma_de_...",,None,2,None,None,FabricaÃ§Ã£o de celulose e outras pastas para ...,3550308,1978-12-06,None,None,2005-11-03,,2046,None,0,,1,10,ATIVA,AVENIDA,SEM MOTIVO,MATRIZ


### DEFs

In [14]:
# metadata silver
FULL_METADATA_MAP = {
    # Identificação Básica
    'cnpj': {'alias': 'NRCNPJ', 'type': 'str'},
    'razao_social': {'alias': 'NMRAZSOC', 'type': 'str'},
    'nome_fantasia': {'alias': 'NMFANT', 'type': 'str'},
    'capital_social': {'alias': 'VLCPTSOC', 'type': 'float64'},
    'identificador_matriz_filial': {'alias': 'IDMTZFIL', 'type': 'int64'},
    'descricao_identificador_matriz_filial': {'alias': 'DSIDMTZFIL', 'type': 'str'},
    
    # Localização
    'uf': {'alias': 'SGUF', 'type': 'str'},
    'cep': {'alias': 'NRCEP', 'type': 'str'},
    'pais': {'alias': 'NMPAIS', 'type': 'str'},
    'codigo_pais': {'alias': 'CDPAIS', 'type': 'str'},
    'municipio': {'alias': 'NMMUN', 'type': 'str'},
    'codigo_municipio': {'alias': 'CDMUN', 'type': 'str'},
    'codigo_municipio_ibge': {'alias': 'CDMUNIBGE', 'type': 'int64'},
    'bairro': {'alias': 'NMBRR', 'type': 'str'},
    'logradouro': {'alias': 'NMLGR', 'type': 'str'},
    'numero': {'alias': 'NREND', 'type': 'str'},
    'complemento': {'alias': 'DSCPL', 'type': 'str'},
    'descricao_tipo_de_logradouro': {'alias': 'DSTIPLGR', 'type': 'str'},
    'nome_cidade_no_exterior': {'alias': 'NMCDEXT', 'type': 'str'},
    
    # Contato
    'email': {'alias': 'DSEML', 'type': 'str'},
    'ddd_fax': {'alias': 'NRFAX', 'type': 'str'},
    'ddd_telefone_1': {'alias': 'NRTEL1', 'type': 'str'},
    'ddd_telefone_2': {'alias': 'NRTEL2', 'type': 'str'},
    
    # Classificação e Natureza
    'cnae_fiscal': {'alias': 'CDCNAEFISC', 'type': 'int64'},
    'cnae_fiscal_descricao': {'alias': 'DSCNAEFISC', 'type': 'str'},
    'natureza_juridica': {'alias': 'DSNATJUR', 'type': 'str'},
    'codigo_natureza_juridica': {'alias': 'CDNATJUR', 'type': 'int64'},
    'porte': {'alias': 'DSPORTE', 'type': 'str'},
    'codigo_porte': {'alias': 'CDPORTE', 'type': 'int64'},
    'qualificacao_do_responsavel': {'alias': 'CDQUALRESP', 'type': 'int64'},
    'ente_federativo_responsavel': {'alias': 'NMENTEFED', 'type': 'str'},
    
    # Situação Cadastral
    'situacao_cadastral': {'alias': 'CDSITCAD', 'type': 'int64'},
    'descricao_situacao_cadastral': {'alias': 'DSSITCAD', 'type': 'str'},
    'data_situacao_cadastral': {'alias': 'DTSITCAD', 'type': 'str'},
    'motivo_situacao_cadastral': {'alias': 'CDMOTSITCAD', 'type': 'int64'},
    'descricao_motivo_situacao_cadastral': {'alias': 'DSMOTSITCAD', 'type': 'str'},
    'situacao_especial': {'alias': 'DSSITESPEC', 'type': 'str'},
    'data_situacao_especial': {'alias': 'DTSITESPEC', 'type': 'str'},
    'data_inicio_atividade': {'alias': 'DTINIATV', 'type': 'str'},
    
    # Simples Nacional e MEI (Campos que causaram o erro)
    'opcao_pelo_simples': {'alias': 'FLSIMPLES', 'type': 'str'}, # Booleano na API vira str aqui para segurança
    'data_opcao_pelo_simples': {'alias': 'DTOPTSIMPLES', 'type': 'str'},
    'data_exclusao_do_simples': {'alias': 'DTEXCSIMPLES', 'type': 'str'},
    'opcao_pelo_mei': {'alias': 'FLMEI', 'type': 'str'},
    'data_opcao_pelo_mei': {'alias': 'DTOPTMEI', 'type': 'str'},
    'data_exclusao_do_mei': {'alias': 'DTEXCMEI', 'type': 'str'},
    
    # QSA (Quadro de Sócios e Administradores)
    'qsa_pais': {'alias': 'QSANMPAIS', 'type': 'str'},
    'qsa_nome_socio': {'alias': 'QSANMSOC', 'type': 'str'},
    'qsa_codigo_pais': {'alias': 'QSACDPAIS', 'type': 'str'},
    'qsa_faixa_etaria': {'alias': 'QSADSFAIXETAR', 'type': 'str'},
    'qsa_cnpj_cpf': {'alias': 'QSANRDOC', 'type': 'str'},
    'qsa_qualificacao_socio': {'alias': 'QSADSQUAL', 'type': 'str'},
    'qsa_codigo_faixa_etaria': {'alias': 'QSACDFAIXETAR', 'type': 'int64'},
    'qsa_data_entrada_sociedade': {'alias': 'QSADTENT', 'type': 'str'},
    'qsa_identificador_de_socio': {'alias': 'QSAIDSOC', 'type': 'int64'},
    'qsa_cpf_representante_legal': {'alias': 'QSANRCPFREP', 'type': 'str'},
    'qsa_nome_representante_legal': {'alias': 'QSANMREP', 'type': 'str'},
    'qsa_codigo_qualificacao_socio': {'alias': 'QSACDQUAL', 'type': 'int64'},
    'qsa_qualificacao_representante_legal': {'alias': 'QSADSQUALREP', 'type': 'str'},
    'qsa_codigo_qualificacao_representante_legal': {'alias': 'QSACDQUALREP', 'type': 'int64'},
    
    # Secundários e Tributário
    'cnaes_secundarios_codigo': {'alias': 'CDCNAESEC', 'type': 'int64'},
    'cnaes_secundarios_descricao': {'alias': 'DSCNAESEC', 'type': 'str'},
    'regime_tributario_ano': {'alias': 'TRIANO', 'type': 'int64'},
    'regime_tributario_cnpj_da_scp': {'alias': 'TRICNJSCP', 'type': 'str'},
    'regime_tributario_forma_de_tributacao': {'alias': 'TRIFORMA', 'type': 'str'},
    'regime_tributario_quantidade_de_escrituracoes': {'alias': 'TRIQTDESCRIT', 'type': 'int64'}
}

RENAME_QSA = {
    'pais': 'qsa_pais', 'nome_socio': 'qsa_nome_socio', 'codigo_pais': 'qsa_codigo_pais',
    'faixa_etaria': 'qsa_faixa_etaria', 'cnpj_cpf_do_socio': 'qsa_cnpj_cpf',
    'qualificacao_socio': 'qsa_qualificacao_socio', 'codigo_faixa_etaria': 'qsa_codigo_faixa_etaria',
    'data_entrada_sociedade': 'qsa_data_entrada_sociedade', 'identificador_de_socio': 'qsa_identificador_de_socio',
    'cpf_representante_legal': 'qsa_cpf_representante_legal', 'nome_representante_legal': 'qsa_nome_representante_legal',
    'tipo_representante_legal': 'qsa_tipo_representante_legal', 'cnpj_cpf_representante_legal': 'qsa_cnpj_cpf_representante_legal',
    'codigo_qualificacao_socio': 'qsa_codigo_qualificacao_socio', 'qualificacao_representante_legal': 'qsa_qualificacao_representante_legal',
    'codigo_qualificacao_representante_legal': 'qsa_codigo_qualificacao_representante_legal'
}

RENAME_CNAE = {'codigo': 'cnaes_secundarios_codigo', 'descricao': 'cnaes_secundarios_descricao'}

RENAME_REGIME = {
    'ano': 'regime_tributario_ano', 'cnpj_da_scp': 'regime_tributario_cnpj_da_scp',
    'forma_de_tributacao': 'regime_tributario_forma_de_tributacao', 
    'quantidade_de_escrituracoes': 'regime_tributario_quantidade_de_escrituracoes'
}

In [37]:
FULL_METADATA_MAP.items()

dict_items([('cnpj', {'alias': 'NRCNPJ', 'type': 'str'}), ('razao_social', {'alias': 'NMRAZSOC', 'type': 'str'}), ('nome_fantasia', {'alias': 'NMFANT', 'type': 'str'}), ('capital_social', {'alias': 'VLCPTSOC', 'type': 'float64'}), ('identificador_matriz_filial', {'alias': 'IDMTZFIL', 'type': 'int64'}), ('descricao_identificador_matriz_filial', {'alias': 'DSIDMTZFIL', 'type': 'str'}), ('uf', {'alias': 'SGUF', 'type': 'str'}), ('cep', {'alias': 'NRCEP', 'type': 'str'}), ('pais', {'alias': 'NMPAIS', 'type': 'str'}), ('codigo_pais', {'alias': 'CDPAIS', 'type': 'str'}), ('municipio', {'alias': 'NMMUN', 'type': 'str'}), ('codigo_municipio', {'alias': 'CDMUN', 'type': 'str'}), ('codigo_municipio_ibge', {'alias': 'CDMUNIBGE', 'type': 'int64'}), ('bairro', {'alias': 'NMBRR', 'type': 'str'}), ('logradouro', {'alias': 'NMLGR', 'type': 'str'}), ('numero', {'alias': 'NREND', 'type': 'str'}), ('complemento', {'alias': 'DSCPL', 'type': 'str'}), ('descricao_tipo_de_logradouro', {'alias': 'DSTIPLGR', '

In [38]:
type_map = {v['alias']: v['type'] for k, v in FULL_METADATA_MAP.items()}
print(type_map)

{'NRCNPJ': 'str', 'NMRAZSOC': 'str', 'NMFANT': 'str', 'VLCPTSOC': 'float64', 'IDMTZFIL': 'int64', 'DSIDMTZFIL': 'str', 'SGUF': 'str', 'NRCEP': 'str', 'NMPAIS': 'str', 'CDPAIS': 'str', 'NMMUN': 'str', 'CDMUN': 'str', 'CDMUNIBGE': 'int64', 'NMBRR': 'str', 'NMLGR': 'str', 'NREND': 'str', 'DSCPL': 'str', 'DSTIPLGR': 'str', 'NMCDEXT': 'str', 'DSEML': 'str', 'NRFAX': 'str', 'NRTEL1': 'str', 'NRTEL2': 'str', 'CDCNAEFISC': 'int64', 'DSCNAEFISC': 'str', 'DSNATJUR': 'str', 'CDNATJUR': 'int64', 'DSPORTE': 'str', 'CDPORTE': 'int64', 'CDQUALRESP': 'int64', 'NMENTEFED': 'str', 'CDSITCAD': 'int64', 'DSSITCAD': 'str', 'DTSITCAD': 'str', 'CDMOTSITCAD': 'int64', 'DSMOTSITCAD': 'str', 'DSSITESPEC': 'str', 'DTSITESPEC': 'str', 'DTINIATV': 'str', 'FLSIMPLES': 'str', 'DTOPTSIMPLES': 'str', 'DTEXCSIMPLES': 'str', 'FLMEI': 'str', 'DTOPTMEI': 'str', 'DTEXCMEI': 'str', 'QSANMPAIS': 'str', 'QSANMSOC': 'str', 'QSACDPAIS': 'str', 'QSADSFAIXETAR': 'str', 'QSANRDOC': 'str', 'QSADSQUAL': 'str', 'QSACDFAIXETAR': 'int6

In [ ]:
BASE_DIR = Path.cwd().parent
RAW_DATA_PATH = os.path.join(BASE_DIR, 'data', 'raw')

def process_raw_to_dataframes(path_name: str) -> dict:
    all_data_frames = {}

    fuso_br = pytz.timezone('America/Sao_Paulo')
    
    alias_map = {k: v['alias'] for k, v in FULL_METADATA_MAP.items()}
    type_map = {v['alias']: v['type'] for k, v in FULL_METADATA_MAP.items()}

    for file in os.listdir(path_name):
        if file.endswith('.json'):
            full_path = os.path.join(path_name, file)

            with open(full_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                df = pd.json_normalize(data)

                def expand_and_rename(df_orig: pd.DataFrame, col_name: str, rename_dict: dict) -> pd.DataFrame:
                    if col_name in df_orig.columns and not df_orig[col_name].dropna().empty:
                        extracted = df_orig[col_name].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else {}) #verify null values
                        df_norm = pd.json_normalize(extracted)
                        return df_norm.rename(columns=rename_dict)
                    return pd.DataFrame()

                df_qsa = expand_and_rename(df, 'qsa', RENAME_QSA)
                df_cnae = expand_and_rename(df, 'cnaes_secundarios', RENAME_CNAE)
                df_regime_trib = expand_and_rename(df, 'regime_tributario', RENAME_REGIME)

                df_final = pd.concat([df, df_qsa, df_cnae, df_regime_trib], axis=1)

                df_final = df_final.rename(columns=alias_map) 
                
                for col, target_type in type_map.items():
                    if col in df_final.columns:
                        if target_type == 'str':
                            df_final[col] = df_final[col].fillna('').astype(target_type)

                        else:
                            df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0).astype(target_type)

                
                df_final = df_final.drop(columns=['qsa', 'cnaes_secundarios', 'regime_tributario'], errors='ignore')
                df_final['DTEXTREF'] = pd.to_datetime(datetime.now(fuso_br).date())

                all_data_frames[file] = df_final
    
    print(len(all_data_frames))

    return all_data_frames

In [40]:
my_dict = process_raw_to_dataframes(RAW_DATA_PATH)

1


In [41]:
for arquivo, df in my_dict.items():
    print(f"Arquivo: {arquivo} | Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")

Arquivo: 19821234000128_2026-03-14.json | Linhas: 1 | Colunas: 65


In [43]:
# Pega o primeiro valor (DataFrame) do dicionário
df_teste = next(iter(my_dict.values()))

# Agora você pode usar o .head()
display(df_teste.head())

,SGUF,NRCEP,NRCNPJ,NMPAIS,DSEML,DSPORTE,NMBRR,NREND,NRFAX,NMMUN,NMLGR,CDCNAEFISC,CDPAIS,DSCPL,CDPORTE,NMRAZSOC,NMFANT,VLCPTSOC,NRTEL1,NRTEL2,FLMEI,CDMUN,DSNATJUR,DSSITESPEC,FLSIMPLES,CDSITCAD,DTOPTMEI,DTEXCMEI,DSCNAEFISC,CDMUNIBGE,DTINIATV,DTSITESPEC,DTOPTSIMPLES,DTSITCAD,NMCDEXT,CDNATJUR,DTEXCSIMPLES,CDMOTSITCAD,NMENTEFED,IDMTZFIL,CDQUALRESP,DSSITCAD,DSTIPLGR,DSMOTSITCAD,DSIDMTZFIL,QSANMPAIS,QSANMSOC,QSACDPAIS,QSADSFAIXETAR,QSANRDOC,QSADSQUAL,QSACDFAIXETAR,QSADTENT,QSAIDSOC,QSANRCPFREP,QSANMREP,QSACDQUAL,QSADSQUALREP,QSACDQUALREP,CDCNAESEC,DSCNAESEC,TRIANO,TRICNJSCP,TRIFORMA,TRIQTDESCRIT
0,SP,04794000,19821234000128,,,DEMAIS,VILA GERTRUDES,14401,,SAO PAULO,DAS NACOES UNIDAS,6462000,,ANDAR 4 - PARTE V CONJ 44 EDIF B1- AROEIRA,5,ODEBRECHT ENGENHARIA E CONSTRUCAO S.A. EM RECU...,,1.064965e+10,1137924000,,,7107,Sociedade Anônima Fechada,RECUPERACAO JUDICIAL,,2,,,Holdings de instituições não-financeiras,3550308,2014-03-05,2024-07-30,,2014-03-05,,2054,,0,,1,64,ATIVA,AVENIDA,SEM MOTIVO,MATRIZ,,RICARDO LUIS MACHADO WEYLL,,Entre 41 a 50 anos,***072565**,Diretor,5,2024-03-14,2,***000000**,,10,Não informada,0,8211300,Serviços combinados de escritório e apoio admi...,2014,,LUCRO REAL,1


In [44]:
df_teste.columns

Index(['SGUF', 'NRCEP', 'NRCNPJ', 'NMPAIS', 'DSEML', 'DSPORTE', 'NMBRR',
       'NREND', 'NRFAX', 'NMMUN', 'NMLGR', 'CDCNAEFISC', 'CDPAIS', 'DSCPL',
       'CDPORTE', 'NMRAZSOC', 'NMFANT', 'VLCPTSOC', 'NRTEL1', 'NRTEL2',
       'FLMEI', 'CDMUN', 'DSNATJUR', 'DSSITESPEC', 'FLSIMPLES', 'CDSITCAD',
       'DTOPTMEI', 'DTEXCMEI', 'DSCNAEFISC', 'CDMUNIBGE', 'DTINIATV',
       'DTSITESPEC', 'DTOPTSIMPLES', 'DTSITCAD', 'NMCDEXT', 'CDNATJUR',
       'DTEXCSIMPLES', 'CDMOTSITCAD', 'NMENTEFED', 'IDMTZFIL', 'CDQUALRESP',
       'DSSITCAD', 'DSTIPLGR', 'DSMOTSITCAD', 'DSIDMTZFIL', 'QSANMPAIS',
       'QSANMSOC', 'QSACDPAIS', 'QSADSFAIXETAR', 'QSANRDOC', 'QSADSQUAL',
       'QSACDFAIXETAR', 'QSADTENT', 'QSAIDSOC', 'QSANRCPFREP', 'QSANMREP',
       'QSACDQUAL', 'QSADSQUALREP', 'QSACDQUALREP', 'CDCNAESEC', 'DSCNAESEC',
       'TRIANO', 'TRICNJSCP', 'TRIFORMA', 'TRIQTDESCRIT'],
      dtype='str')

In [45]:
for c, i in df_teste.dtypes.items():
    print(c, i)

SGUF str
NRCEP str
NRCNPJ str
NMPAIS str
DSEML str
DSPORTE str
NMBRR str
NREND str
NRFAX str
NMMUN str
NMLGR str
CDCNAEFISC int64
CDPAIS str
DSCPL str
CDPORTE int64
NMRAZSOC str
NMFANT str
VLCPTSOC float64
NRTEL1 str
NRTEL2 str
FLMEI str
CDMUN str
DSNATJUR str
DSSITESPEC str
FLSIMPLES str
CDSITCAD int64
DTOPTMEI str
DTEXCMEI str
DSCNAEFISC str
CDMUNIBGE int64
DTINIATV str
DTSITESPEC str
DTOPTSIMPLES str
DTSITCAD str
NMCDEXT str
CDNATJUR int64
DTEXCSIMPLES str
CDMOTSITCAD int64
NMENTEFED str
IDMTZFIL int64
CDQUALRESP int64
DSSITCAD str
DSTIPLGR str
DSMOTSITCAD str
DSIDMTZFIL str
QSANMPAIS str
QSANMSOC str
QSACDPAIS str
QSADSFAIXETAR str
QSANRDOC str
QSADSQUAL str
QSACDFAIXETAR int64
QSADTENT str
QSAIDSOC int64
QSANRCPFREP str
QSANMREP str
QSACDQUAL int64
QSADSQUALREP str
QSACDQUALREP int64
CDCNAESEC int64
DSCNAESEC str
TRIANO int64
TRICNJSCP str
TRIFORMA str
TRIQTDESCRIT int64


#### f2 SILVER

In [46]:
BASE_DIR = Path.cwd().parent
SILVER_DATA_PATH = os.path.join(BASE_DIR, 'data', 'silver')

def save_to_silver( dfs_dict : dict) -> None:
    os.makedirs(SILVER_DATA_PATH, exist_ok=True) #create silver directory if it doesn't exist

    fuso_br = pytz.timezone('America/Sao_Paulo') #timezone of Brazil
    today = datetime.now(fuso_br).strftime('%Y-%m-%d') #today's date

    for file_name, df in dfs_dict.items(): #iterate over the dictionary
        if df.empty:
            continue #error handling

        cnpj_prefix = file_name.split('_')[0] #extract cnpj prefix
        file_path = os.path.join(SILVER_DATA_PATH, f"{cnpj_prefix}_{today}.parquet") #parquet file path
        try:
            df.to_parquet(file_path, index=False) #save to parquet
            print(f"File saved successfully at: {file_path}")
        except Exception as e:
            print(f"Error saving file {file_path}: {e}")

In [47]:
save_to_silver(my_dict)

File saved successfully at: c:\Users\annab\beatrizbeserra\PROJETOS\GITHUB\Portifolio\projetos\data_engineering\credit_guard_ml_pipeline\data\silver\19821234000128_2026-03-14.parquet


#### SENDING TO BIGQUERY

In [26]:
import pandas as pd
import os
from google.cloud import storage
from google.cloud import bigquery

In [28]:
# Authetication (just 1 time)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../key-google.json'
client_st = storage.Client()
client_bq = bigquery.Client()
bucket = client_st.bucket('credit-guard-raw-sa-east1')

for blob in bucket.list_blobs():
    print(blob.name)

raw/cnpj/ingestion_date=2026-02-06/00000000000191_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/00001180000126_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/00360305000104_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/00776574000156_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/00864214000106_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/01027058000191_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/01637895000132_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/02012862000160_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/02332886000104_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/02429144000193_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/02916265000160_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/03220438000173_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/03361252000134_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/03853896000140_2026-02-06.json
raw/cnpj/ingestion_date=2026-02-06/05197443000138_2026-02-06.json
raw/cnpj/i

In [ ]:
DATASET_ID = 'credit_guard_dataset_pipeline'
TABLE_ID = f"{client_bq.project}.{DATASET_ID}.credit_guard_cnpj_silver"

try:
    if DATASET_ID not in [dataset.dataset_id for dataset in client_bq.list_datasets()]:
        client_bq.create_dataset(DATASET_ID, exists_ok=True)
except Exception as e:
    print(f"Error creating dataset: {e}")


job_config = bigquery.LoadJobConfig(
    source_format = bigquery.SourceFormat.PARQUET,
    write_disposition = 'WRITE_TRUNCATE',
    autodetect = True,
)

source_uri = "gs://credit-guard-raw-sa-east1/silver/*.parquet"
print(f"Iniciating data charging {source_uri}...")

load_job = client_bq.load_table_from_uri(
    source_uri,
    TABLE_ID,
    job_config=job_config
)

load_job.result()
print(f"Charging ended successfully! Table {TABLE_ID} updated.")

# --- Additional logs with error handler for parcial errors ---
if load_job.errors:
    print("⚠️  Warning: Some rows failed to load:")
    for error in load_job.errors:
        print(f" - {error['message']}")
else:
    print("✅ No errors reported during the load job.")

destination_table = client_bq.get_table(TABLE_ID)  # Faz uma chamada rápida para ler os metadados da tabela

print(f"--- Job Summary ---")
print(f"Job ID: {load_job.job_id}")
print(f"Status: {load_job.state}")
print(f"Rows loaded: {destination_table.num_rows}") # Total lines after updated
print(f"Total size: {destination_table.num_bytes / 1024**2:.2f} MB")
print(f"Table {TABLE_ID} updated successfully!")

Iniciating data charging gs://credit-guard-raw-sa-east1/silver/*.parquet...
Charging ended successfully! Table project-626d9b15-4a46-44bb-936.credit_guard_dataset_pipeline.credit_guard_cnpj_silver updated.
